In [ ]:
import zipfile
!wget https://storage.googleapis.com/ztm_tf_course/food_vision/pizza_steak.zip

In [ ]:
!unzip /content/pizza_steak.zip

In [ ]:
# zip_ref=zipfile.ZipFile('/content/pizza_steak.zip','r')
# zip_ref.extractall()
# zip_ref.close()

In [ ]:
# with zipfile.ZipFile('/content/pizza_steak.zip','r') as zip_ref:
#   zip_ref.extractall()

In [ ]:
!ls /content/pizza_steak-- #icinde olanlari gosterir

In [ ]:
import os
for dirpath,dirnames,filenames in os.walk('/content/pizza_steak'):
  print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}.")

In [ ]:
num_steak_images_train=len(os.listdir('/content/pizza_steak/train/steak'))
num_steak_images_train

In [ ]:


import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random
import pathlib

def view_random_image(target_dir,target_class):
  target_folder=pathlib.Path(target_dir)/target_class
  random_image=random.sample(os.listdir(target_folder),1)
  path=target_folder /random_image[0]
  img=mpimg.imread(path)
  plt.imshow(img)
  plt.title(target_class)
  plt.axis('off')
  print(f"image shape: {img.shape}")

In [ ]:
view_random_image(target_dir='/content/pizza_steak/train/',target_class='pizza')

In [ ]:
import tensorflow as tf
IMG_SIZE=(224,224)

train_dir='/content/pizza_steak/train/'
test_dir='/content/pizza_steak/test/'

train_data=tf.keras.preprocessing.image_dataset_from_directory(train_dir,label_mode='binary',batch_size=32,
                                                               image_size=IMG_SIZE,shuffle=True,crop_to_aspect_ratio=True)

test_data=tf.keras.preprocessing.image_dataset_from_directory(test_dir,image_size=IMG_SIZE,label_mode='binary',crop_to_aspect_ratio=True)

In [ ]:
model=tf.keras.Sequential([
    tf.keras.layers.Input(shape=[224,224,3]),
    tf.keras.layers.Rescaling(1/255.),
    tf.keras.layers.Conv2D(filters=64,kernel_size=7,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(pool_size=2),
    tf.keras.layers.Conv2D(filters=128,kernel_size=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.Conv2D(filters=128,kernel_size=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(pool_size=2),
    tf.keras.layers.Conv2D(filters=256,kernel_size=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.Conv2D(filters=256,kernel_size=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(pool_size=2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=128,activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=64,activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=1,activation='sigmoid')
])

In [ ]:
model.compile(loss='binary_crossentropy',optimizer=tf.keras.optimizers.SGD(learning_rate=0.001,momentum=0.9),metrics=['accuracy'])

model.fit(train_data,validation_data=test_data,epochs=5)

In [ ]:
model.evaluate(train_data)

In [ ]:
model.evaluate(test_data)

In [ ]:
data_augementation=tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomHeight(0.1),
    tf.keras.layers.RandomWidth(0.1)
])

In [ ]:
class_names=train_data.class_names
class_names

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12,10))
for images,labels in train_data.take(1):
  augmented_images=data_augementation(images)
  for i in range(12):
    ax=plt.subplot(3,4,i+1)
    plt.imshow(augmented_images[i].numpy().astype('uint8'))
    plt.title(class_names[labels[i].numpy().astype('int8')[0]])
    plt.axis('off')

In [ ]:
model=tf.keras.Sequential([
    tf.keras.layers.Input(shape=[224,224,3]),
    data_augementation,
    tf.keras.layers.Rescaling(1/255.),
    tf.keras.layers.Resizing(224,224),
    tf.keras.layers.Conv2D(filters=64,kernel_size=7,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(pool_size=2),
    tf.keras.layers.Conv2D(filters=128,kernel_size=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.Conv2D(filters=128,kernel_size=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(pool_size=2),
    tf.keras.layers.Conv2D(filters=256,kernel_size=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.Conv2D(filters=256,kernel_size=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(pool_size=2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=128,activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=64,activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=1,activation='sigmoid')
])

In [ ]:
model.compile(loss='binary_crossentropy',optimizer=tf.keras.optimizers.SGD(learning_rate=0.003,momentum=0.9),metrics=['accuracy'])

In [ ]:
model.fit(train_data,validation_data=test_data,epochs=5)

In [ ]:
base_model=tf.keras.applications.ResNet50(include_top=False)
avg=tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output=tf.keras.layers.Dense(units=1,activation='sigmoid')(avg)
model=tf.keras.Model(inputs=base_model.input,outputs=output)

In [ ]:
base_model.training=False

In [ ]:
initial_learning_rate=0.01
lr_schedule=tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate,
    decay_steps=47,
    decay_rate=0.96
)

early_stopping=tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)

model.compile(loss='binary_crossentropy',optimizer=tf.keras.optimizers.SGD(learning_rate=lr_schedule,momentum=0.9),metrics=['accuracy'])

model.fit(train_data,epochs=15,validation_data=test_data,callbacks=early_stopping)

Epoch 1/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.8566 - loss: 0.3336 - val_accuracy: 0.8400 - val_loss: 0.7131
Epoch 2/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 29s 302ms/step - accuracy: 0.9653 - loss: 0.0811 - val_accuracy: 0.9540 - val_loss: 0.1092
Epoch 3/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 16s 340ms/step - accuracy: 0.9953 - loss: 0.0210 - val_accuracy: 0.9780 - val_loss: 0.0553
Epoch 4/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 19s 304ms/step - accuracy: 0.9928 - loss: 0.0181 - val_accuracy: 0.9860 - val_loss: 0.0491
Epoch 5/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 20s 296ms/step - accuracy: 0.9972 - loss: 0.0057 - val_accuracy: 0.9860 - val_loss: 0.0582
Epoch 6/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 22s 329ms/step - accuracy: 1.0000 - loss: 0.0012 - val_accuracy: 0.9860 - val_loss: 0.0448
Epoch 7/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 16s 336ms/step - accuracy: 1.0000 - loss: 5.8117e-04 - val_accuracy: 0.9900 - val_loss: 0.0448
Epoch 8/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 14s 301ms/step - accuracy: 1.0000 - loss: 8.9557e-04 - val

In [ ]:
inputs=tf.keras.layers.Input(shape=(224,224,3))

x=data_augementation(inputs)

x=tf.keras.applications.resnet50.preprocess_input(x)
base_model=tf.keras.applications.ResNet50(include_top=False,weights='imagenet',input_tensor=x)
avg=tf.keras.layers.GlobalAveragePooling2D()(base_model)
output=tf.keras.layers.Dense(units=1,activation='sigmoid')(avg)
model=tf.keras.Model(inputs=inputs,outputs=output)

ValueError: Only input tensors may be passed as positional arguments. The following argument value should be passed as a keyword argument: <Functional name=resnet50, built=True> (of type <class 'keras.src.models.functional.Functional'>)

In [ ]:
base_model.training=False

In [ ]:
initial_learning_rate=0.01
lr_schedule=tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate,
    decay_steps=47,
    decay_rate=0.96
)

early_stopping=tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)

model.compile(loss='binary_crossentropy',optimizer=tf.keras.optimizers.SGD(learning_rate=lr_schedule,momentum=0.9),metrics=['accuracy'])

model.fit(train_data,epochs=15,validation_data=test_data,callbacks=early_stopping)
